# 蜃景 × lightx2v —— 纯文生视频 (t2v) Colab 部署（无 ComfyUI）

**怎么用：菜单「代码执行程序 → 全部运行」(Run all)。** 断线/回收后再 Run all。
- **Wan 权重(67G)直接下到本地 `/content`**（HF CDN ~170MB/s，比 Drive FUSE 读快几倍；本地 SSD 加载秒级）。代价=每会话重下(~10 分钟)，但整体比挂 Drive 加载快。
- lightx2v 每会话重装（很快，`--no-deps`）；SageAttention 现编译（可选，几分钟）。

**跑前一次性：**
1. 运行时选 **GPU**（Blackwell / Hopper / Ada / A100 都行）。
2. 右侧 🔑 Secrets 可加 **HF_TOKEN**（下 Wan 权重快些；公开模型匿名也能下）。
3. 第 1 格 `DEEPSEEK_KEY` 可留空（分镜 / AI 分析用前端配的 grok 模型）。

**§3/§4 已把 2026-06-17 实战踩平的坑固化（不用再手动改）：**
- lightx2v 0.1.0 钉死 `torch<2.8.0` 撞 cu128 torch 2.8 → 本体&依赖全 `--no-deps` 装。
- torch↔torchvision ABI 不匹配 → 子进程探测、坏了原子重装匹配的 cu128 三件套。
- `worldmirror`/`ring_attn` 跟单卡 t2v 无关却崩 import → 自动 patch 文件。
- editable `.pth` 内核不重读 → 一切**以干净子进程验证**。
- 配置**必须用 `configs/wan22/wan_moe_t2v.json`**（含 `boundary=0.875` 双专家切换点）；`wan/wan_t2v.json` 是旧单模型会崩 `KeyError 'boundary'`。§3 自动选对。
- 注意力默认 `torch_sdpa`（稳）；配置里默认的 `flash_attn3` 没装、调到会 `'NoneType' not callable` 崩，§3 已改掉；SageAttention 编上了用 `sage_attn2`（快）。
- **★`rope_type` 默认 `flashinfer`（没装）也会 `'NoneType' not callable`** → §3 已强制 `rope_type=torch`（出片跑通的最后一把钥匙，RTX PRO 6000 实测）。

**runner** = `wan2.2_moe`（A14B 双专家），`cpu_offload` 按显存自动（RTX PRO 6000 96G 实测全 GPU bf16；5090 32G 自动卸载）。**前端纯 t2v**：粘小说 → 一键分析 → 每镜「出片(t2v)」。

In [ ]:
# === §1 参数 + Drive + GPU 探测 ===
import os, sys
DEEPSEEK_KEY = ''            # 分镜/AI 分析用;留空=用前端配的 grok
REPO_URL = 'https://github.com/aw3n1998/build-with-langchain.git'; BRANCH = 'main'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'; CACHE = DRIVE + '/mirage_models'; TOOLS = DRIVE + '/mirage_tools'
for p in (CACHE, TOOLS):
    os.makedirs(p, exist_ok=True)
os.environ['HF_HOME'] = TOOLS + '/hf_cache'; os.makedirs(os.environ['HF_HOME'], exist_ok=True)
!nvidia-smi -L
import torch
if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    cc = torch.cuda.get_device_capability(0)
    os.environ['LX_CPU_OFFLOAD'] = 'false' if gb > 70 else 'true'   # >70G 两专家全 GPU;否则挪 CPU
    print(f'GPU {torch.cuda.get_device_name(0)} sm{cc[0]}.{cc[1]} {gb:.0f}G -> cpu_offload={os.environ["LX_CPU_OFFLOAD"]}')
    if cc[0] >= 12:
        print('  Blackwell(sm120): §3 编译 SageAttention(官方含 12.0)、patch worldmirror。')
else:
    os.environ['LX_CPU_OFFLOAD'] = 'true'; print('没检测到 GPU! 更改运行时类型 -> 选 GPU')
os.environ['PYTHONPATH'] = os.pathsep.join(p for p in sys.path if p)   # 子进程继承内核 torch
print('环境就绪 | 模型:', CACHE, '| 工具:', TOOLS)

In [ ]:
# === §2 拉取/更新 mirage 仓库 ===
import os, sys
%cd /content
if not os.path.isdir('mirage/.git'):
    !git clone -b {BRANCH} {REPO_URL} mirage
else:
    !cd mirage && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
%cd /content/mirage
sys.path.insert(0, '/content/mirage/colab')
print('mirage 就绪 @', BRANCH)

In [ ]:
# === §3 装 lightx2v（torch 锁 cu128 + --no-deps 绕 torch<2.8.0 冲突 + patch worldmirror/ring_attn + 选对 MoE 配置）===
# 今日实战踩平的坑(详见 git log / 笔记 lightx2v-t2v-blackwell-install):
#   ① lightx2v 0.1.0 钉 torch<2.8.0 撞 cu128-torch2.8 → 本体&依赖全 --no-deps 装;② 残留元数据→装前 uninstall;
#   ③ editable .pth 内核不重读→sys.path 加源码;④ worldmirror/ring_attn 崩 import→patch 文件;⑤ torch↔tv ABI→原子重装;
#   ⑥ 配置:必须用 configs/wan22/wan_moe_t2v.json(含 boundary=0.875 双专家切换点),configs/wan/wan_t2v.json 是旧单模型会崩 KeyError 'boundary'。
# 心法:一切以「干净子进程」import 为准(内核会被装卸污染;§5 server 本就是子进程)。
import os, subprocess, sys, json, re, glob, importlib.util as u
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout
def _ok(mod): return 'Traceback' not in sh(f'python -c "import {mod}" 2>&1')   # 子进程 import 成功=无 Traceback
LX = '/content/LightX2V'
# 1) clone + 源码加进 sys.path(editable 的 .pth 当前内核不重读)
if not os.path.isdir(LX + '/.git'):
    sh(f'git clone https://github.com/ModelTC/LightX2V.git {LX}')
if LX not in sys.path: sys.path.insert(0, LX)
# 2) torch 三件套:缺了或 torchvision ABI 坏了(子进程探测)→ 一条命令原子重装匹配的 cu128 套
if not _ok('torchvision'):
    print('torch/torchvision 缺或 ABI 不匹配 → 原子重装匹配的 cu128 三件套...')
    print(sh('pip install -q --force-reinstall --no-deps torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3'))
v = sh("python -c 'import torch,torchvision,torchaudio;print(torch.__version__);print(torchvision.__version__);print(torchaudio.__version__)'").split()
con = '/content/torch_con.txt'; open(con, 'w').write('\n'.join(f'{p}=={x}' for p, x in zip(('torch', 'torchvision', 'torchaudio'), v[:3])))
print('torch 锁定:', v)
# 3) patch 跟单卡 t2v 无关、却崩 import 的两处(改文件→server 子进程也生效;幂等)
fa = LX + '/lightx2v/models/networks/worldmirror/models/layers/attention.py'
if os.path.exists(fa):
    s = open(fa).read()
    s = s.replace('from flash_attn_interface import flash_attn_func as flash_attn_func_v3', 'flash_attn_func_v3 = None  # [patch]')
    s = s.replace('from flash_attn.flash_attn_interface import flash_attn_func as flash_attn_func_v2', 'flash_attn_func_v2 = None  # [patch]')
    open(fa, 'w').write(s)
fr = LX + '/lightx2v/common/ops/attn/ring_attn.py'
if os.path.exists(fr):
    s = open(fr).read()
    s = re.sub(r'@torch\.jit\.script', '# @torch.jit.script  # [patch]', s)
    s = re.sub(r'torch\.jit\.script\(\s*([A-Za-z_]\w*)\s*\)', r'\1', s)
    open(fr, 'w').write(s)
# 4) 装 lightx2v(子进程 import 不过才装):本体 --no-deps 绕 torch<2.8.0 冲突;依赖也 --no-deps 装
if not _ok('lightx2v'):
    sh('pip uninstall -y lightx2v')
    print('装 lightx2v 本体(--no-deps,绕 torch<2.8.0 冲突)...'); print(sh(f'cd {LX} && pip install -e . --no-deps 2>&1 | tail -3'))
    import importlib.metadata as md
    try: reqs = md.requires('lightx2v') or []
    except Exception: reqs = []
    deps = [r.split(';')[0].strip() for r in reqs if 'extra ==' not in r and 'extra==' not in r]
    deps = [d for d in deps if d and not d.lower().startswith('torch')]
    open('/content/lx_deps.txt', 'w').write('\n'.join(deps))
    print(f'补装依赖({len(deps)} 个,--no-deps -c con)...'); print(sh(f'pip install -r /content/lx_deps.txt --no-deps -c {con} 2>&1 | tail -3'))
    NAME = {'cv2': 'opencv-python', 'PIL': 'pillow', 'yaml': 'pyyaml', 'sklearn': 'scikit-learn', 'skimage': 'scikit-image'}
    for _ in range(15):
        err = sh('python -c "import lightx2v" 2>&1')
        if 'Traceback' not in err: break
        m = re.search(r"No module named '([\w.]+)'", err)
        if not m: print('⚠ import 非缺包错:', err[-300:]); break
        mod = m.group(1).split('.')[0]; pkg = NAME.get(mod, mod); print('缺', mod, '→装', pkg); sh(f'pip install {pkg} --no-deps -c {con}')
# 5) SageAttention(可选提速;失败自动退 torch_sdpa)
if not _ok('sageattention'):
    if not os.path.isdir('/content/SageAttention/.git'):
        sh('git clone https://github.com/thu-ml/SageAttention.git /content/SageAttention')
    print('编译 SageAttention(sm120,约 3-8 分钟;失败退 torch_sdpa)...')
    print(sh(f'cd /content/SageAttention && CUDA_ARCHITECTURES="8.0,8.6,8.9,9.0,12.0" EXT_PARALLEL=4 '
             f'NVCC_APPEND_FLAGS="--threads 8" MAX_JOBS=32 pip install -e . --no-deps -c {con} 2>&1 | tail -5'))
_sage = _ok('sageattention')
# 6) 选对 MoE 配置(含 boundary)+ 套注意力/显存策略,写出 /content 供 §5 用(★兼容 5090:小卡 cpu_offload+model级)
_cands = sorted(set(glob.glob(f'{LX}/configs/**/wan_moe_t2v*.json', recursive=True)))
src_cfg = next((c for c in _cands if 'boundary' in json.load(open(c)) and 'lora' not in c.lower() and 'distill' not in c.lower()), None) \
       or next((c for c in _cands if 'boundary' in json.load(open(c))), None) or f'{LX}/configs/wan/wan_t2v.json'
c = json.load(open(src_cfg))
_attn = 'sage_attn2' if _sage else 'torch_sdpa'
for k in ('self_attn_1_type', 'cross_attn_1_type', 'cross_attn_2_type'): c[k] = _attn
_off = (os.environ.get('LX_CPU_OFFLOAD', 'true') == 'true')   # §1 按显存:>70G=False(双专家全GPU)/≤70G=True(卸载)
c['cpu_offload'] = _off
if _off:   # 小卡(5090 32G):cpu_offload + 模型级粒度;仍 OOM 把 offload_granularity 改 'block'→'phase'
    c['offload_granularity'] = 'model'; c['offload_ratio'] = 1.0
    c['t5_cpu_offload'] = False; c['vae_cpu_offload'] = False
else:
    c['offload_granularity'] = 'block'
# fp8(可选默认关):lightx2v 无现成 T2V-A14B fp8 权重,要 tools/convert/converter.py --linear_type fp8 自转双专家+sgl-kernel,
#   且 merge LoRA 与量化互斥(挂人物 LoRA 必须 bf16)。开法:c['dit_quantized']=True; c['dit_quant_scheme']='fp8-sgl'; c['dit_quantized_ckpt']='<fp8目录>'
c['rope_type'] = 'torch'        # ★出片 NoneType 真凶:默认 flashinfer 没装(--no-deps)→apply_rope=None→调它崩;强制 torch
c['rms_norm_type'] = 'torch'    # 防御:默认可能选未装的 sgl-kernel,强制 torch 稳
# ★出片时长:把帧长写进 config(治 config-only 那种 server「调帧数还出5秒」);只覆盖 config 已有的帧长键(不新增未知键,免 server 校验失败)。
_NF = int(os.environ.get('LIGHTX2V_NUM_FRAMES', '81') or 81)   # 想默认更长改这个(81≈5s/121≈7.5s/161≈10s,须 4n+1)
_nfk = [k for k in ('target_video_length', 'num_frames', 'video_length', 'num_frame', 'target_frames') if k in c]
for _k in _nfk: c[_k] = _NF
print('帧长键:', _nfk or '(config 无帧长键 → 帧数走 per-request)', '→', _NF, '帧')
# 防御:sample_guide_scale 必须是 2 元素列表(MoE 双专家各取[0]/[1]);base 若是标量,裸片 §5 第一帧就崩 'float' object is not subscriptable。
_sg = c.get('sample_guide_scale')
if not isinstance(_sg, (list, tuple)):
    _g = float(_sg) if _sg is not None else 5.0; c['sample_guide_scale'] = [_g, _g]
USE_CFG = '/content/wan_moe_t2v_use.json'; json.dump(c, open(USE_CFG, 'w'), indent=2)
os.environ['LX_CONFIG'] = USE_CFG
# 7) 终检(干净子进程为准=§5 server 启动环境)
print('—' * 10)
print('lightx2v import:', '✅ OK' if _ok('lightx2v') else '❌ 看上面报错', '| 配置:', src_cfg.replace(LX + '/', ''),
      '| boundary:', c.get('boundary'), '| SageAttention:', _sage, '| attn:', _attn, '| cpu_offload:', _off)


In [ ]:
# === §4 下权重（有就用、没有才下：本地优先→Drive 其次→都没有才下到 Drive 持久，下次免重下）===
# 2026-06-19 改:之前固定下本地 /content(每会话重下);现在【Drive 有就直接用、不再重下】(代价:从 Drive 加载比本地 SSD 略慢)。
#   想要本地秒载就手动把权重放到 /content/wan_local(本格会优先用它)。LoRA 小,仍放 Drive 持久。
import os, glob
from huggingface_hub import snapshot_download
LORA = CACHE + '/lightx2v_t2v_lora'; os.makedirs(LORA, exist_ok=True)
_LOCAL = '/content/wan_local'; _DRIVE = CACHE + '/wan2.2_t2v_a14b'   # 本地秒载但每会话重下;Drive 持久但加载略慢
def _complete(d): return len(glob.glob(d + '/high_noise_model/*.safetensors')) == 6 and len(glob.glob(d + '/low_noise_model/*.safetensors')) == 6
_PAT = ['high_noise_model/*', 'low_noise_model/*', '*.json', 'Wan2.1_VAE.pth', 'models_t5_umt5-xxl-enc-bf16.pth', 'google/*']
if _complete(_LOCAL):
    MODEL = _LOCAL; print('本地已有完整权重,跳过下载(秒载)。')
elif _complete(_DRIVE):
    MODEL = _DRIVE; print('★Drive 已有完整权重 → 直接用 Drive,不重下。')
else:
    MODEL = _DRIVE; os.makedirs(MODEL, exist_ok=True)   # ★下到 Drive 持久:下次任何会话都不再重下
    print('Drive 也没有 → 从 HF 下 Wan2.2-T2V-A14B 到 Drive 持久(~67G,并行8线程,断点续传;只此一次,下次免下)...')
    snapshot_download('Wan-AI/Wan2.2-T2V-A14B', local_dir=MODEL, max_workers=8, allow_patterns=_PAT)
# 4步蒸馏 LoRA(小,提速必挂)→Drive 持久
_LORA = 'Wan2.2-T2V-A14B-4steps-lora-rank64-Seko-V2.0'
snapshot_download('lightx2v/Wan2.2-Lightning', local_dir=LORA, allow_patterns=[f'{_LORA}/*'])
# 蒸馏 LoRA 双专家 high/low 路径 → 环境变量(供 §5d 挂载 + §6 .env);文件名 glob 实际为准(别写死)
def _find1(d, kw):
    fs = sorted(glob.glob(f'{d}/*{kw}*.safetensors')) or sorted(glob.glob(f'{d}/**/*{kw}*.safetensors', recursive=True))
    return fs[0] if fs else ''
_dld = f'{LORA}/{_LORA}'
os.environ['LIGHTX2V_DISTILL_LORA_HIGH'] = _find1(_dld, 'high')
os.environ['LIGHTX2V_DISTILL_LORA_LOW'] = _find1(_dld, 'low')
hi = len(glob.glob(f'{MODEL}/high_noise_model/*.safetensors')); lo = len(glob.glob(f'{MODEL}/low_noise_model/*.safetensors'))
os.environ['LIGHTX2V_MODEL_T2V'] = MODEL
print('本地权重 high/low:', hi, '/', lo, '✅' if (hi == 6 and lo == 6) else '❌ 缺→重跑本格(续传)', '| MODEL =', MODEL)
print('蒸馏 LoRA high/low:', os.environ['LIGHTX2V_DISTILL_LORA_HIGH'] or '(未找到→检查仓库结构)', '/', os.environ['LIGHTX2V_DISTILL_LORA_LOW'] or '(未找到)')

In [ ]:
# === §5 起 lightx2v server（model_cls=wan2.2_moe；端口 8189；脱离内核,停 cell 不杀）===
import os, subprocess, sys, glob, json
sys.path.insert(0, '/content/mirage/colab'); import persist
LX = '/content/LightX2V'
# 权重源:优先本地(§4 下到本地=秒载),否则退 Drive
_local = '/content/wan_local'
MODEL = _local if (len(glob.glob(_local + '/high_noise_model/*.safetensors')) == 6 and len(glob.glob(_local + '/low_noise_model/*.safetensors')) == 6) else CACHE + '/wan2.2_t2v_a14b'
_isloc = (MODEL == _local)
# 配置:用 §3 选好的 MoE 配置(含 boundary;§3 写到 LX_CONFIG);兜底自己找一个含 boundary 的
CFG = os.environ.get('LX_CONFIG', '')
if not (CFG and os.path.exists(CFG)):
    CFG = next((c for c in sorted(glob.glob(f'{LX}/configs/**/wan_moe_t2v*.json', recursive=True)) if 'boundary' in json.load(open(c))), f'{LX}/configs/wan22/wan_moe_t2v.json')
# ★把出片真帧挖出来:lightx2v server base.py 的 `raise exc` 是重抛(真错那帧被藏)→在重抛前 dump 完整 traceback。
#   幂等;专治「出片报 'NoneType' object is not callable 但看不到哪行 None」。重起后出片即可 grep REAL_TB /content/lightx2v.log。
for _bp in glob.glob(LX + '/lightx2v/server/**/base.py', recursive=True):
    _s = open(_bp).read()
    if 'REAL_TB' not in _s and 'raise exc' in _s:
        open(_bp, 'w').write(_s.replace('raise exc',
            'import traceback as _t; print("REAL_TB\\n"+"".join(_t.format_exception(type(exc), exc, exc.__traceback__)), flush=True); raise exc', 1))
        print('[patch] 出片真 traceback 已插桩:', _bp.replace(LX + '/', ''))
subprocess.run("fuser -k 8189/tcp 2>/dev/null; pkill -9 -f 'lightx2v.server' 2>/dev/null; true", shell=True)  # 硬重启
e = dict(os.environ); e['CUDA_VISIBLE_DEVICES'] = '0'; e['PYTHONUNBUFFERED'] = '1'
open('/content/lightx2v.log', 'w').close()
subprocess.Popen(['python', '-u', '-m', 'lightx2v.server', '--model_cls', 'wan2.2_moe', '--task', 't2v',
                  '--model_path', MODEL, '--config_json', CFG,
                  '--host', '0.0.0.0', '--port', '8189'],
                 cwd=LX, env=e, stdout=open('/content/lightx2v.log', 'w'), stderr=subprocess.STDOUT, start_new_session=True)
print('lightx2v server 起中 | 权重:', '本地 SSD(秒载) ✅' if _isloc else 'Drive(慢)', '| 配置:', CFG.replace(LX + '/', ''))
ok = persist.wait_http('http://127.0.0.1:8189/v1/service/status', timeout=(240 if _isloc else 900))
print('✅ lightx2v 就绪' if ok else '✗ 还没就绪 → 别重跑本格(会重载)!跑 §5c 查进度')
print(subprocess.run('tail -40 /content/lightx2v.log', shell=True, capture_output=True, text=True).stdout)
print('\n★崩 KeyError boundary → 配置用错了(该用 configs/wan22/wan_moe_t2v.json,不是 wan/wan_t2v.json);崩 patch_embedding → 跑 §5b。')
print('★裸片(无 LoRA)先验证通;出片报 NoneType → grep REAL_TB /content/lightx2v.log 看真帧。裸片通了再跑 §5d 挂 LoRA。')

In [ ]:
# === §5b (仅当 §5 崩在「KeyError patch_embedding / 加载权重」时跑) diffusers→native 转格式 ===
# ★Run all 默认【跳过】本格!转格式要几分钟、且往模型目录写文件——只有 §5 真崩 KeyError patch_embedding 才需要。
# 要跑本格:先在新格执行  os.environ["RUN_5B_CONVERT"]="1"  再单独跑本格。
import os, glob
if os.environ.get('RUN_5B_CONVERT') != '1':
    print('§5b 已跳过(Run all 默认)。只有 §5 崩在 KeyError patch_embedding 时才需:按本格顶部注释设环境变量后单独跑。')
else:
    LX = '/content/LightX2V'; MODEL = '/content/wan_local'   # §4 下到本地的权重
    for d in ('high_noise_model', 'low_noise_model'):
        !cd {LX} && python tools/convert/converter.py --source {MODEL}/{d}/ --output {MODEL}/{d}/ --output_ext .safetensors --output_name wan_{d} --model_type wan_dit --single_file
    f = glob.glob(f'{MODEL}/high_noise_model/wan_*.safetensors')
    if f:
        from safetensors import safe_open
        print('转后 key 样例:', list(safe_open(f[0], 'pt').keys())[:5])
    print('转完 → 回 §5 重起 server')

In [ ]:
# === §5c 查 server 状态（不重启！server 加载 67G 慢,用这个看进度,别重跑 §5）===
import subprocess
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout
alive = bool(sh("pgrep -f 'lightx2v.server'").strip())
st = sh("curl -s -m 5 http://127.0.0.1:8189/v1/service/status") or '(还没响应,可能仍在加载)'
print('① server 进程:', '活着 ✅' if alive else '没了 ❌(崩了→看下面日志尾有无 Traceback)')
print('② status 端点:', st)
print('③ 日志尾 30 行:')
print(sh('tail -30 /content/lightx2v.log'))
print('\n判断: 进程活+status有响应=就绪(去 §6/§8/§9);进程活+无响应+日志在动无Traceback=还在加载(等几分钟再跑本格);进程没了+有Traceback=崩了。')

In [ ]:
# === §5d 挂角色/蒸馏 LoRA → 重起 server（自动找角色LoRA→存Drive + 先腾显存防OOM + 起来后自检key）===
# lightx2v 的 LoRA 只认「起 server 的 config」(per-request 会被忽略) → 改 LoRA/帧数/步数都要跑本格重起。
# ★挂 LoRA 必须 bf16(本格自动关量化)。★训练(run.py)与 server 不能同时占卡 → 本格先杀训练腾显存(防 T5 OOM)。
import os, json, subprocess, glob, sys, shutil
sys.path.insert(0, '/content/mirage/colab'); import persist
LX = '/content/LightX2V'
LORA_DIR = os.environ.get('COMFYUI_LORA_DIR') or '/content/drive/MyDrive/mirage_models/char_loras'
os.makedirs(LORA_DIR, exist_ok=True)

# 1) 自动找角色 LoRA:多目录(char_loras / ComfyUI / 训练输出)+两种命名(*_wan_t2v_high / *_high_noise)
#    +排除蒸馏(high_noise_model)和中途检查点(带步数)+取最新 final。找到就拷到 Drive 持久(免会话回收丢)。
#    想手动指定:跑本格前 os.environ['WAN_T2V_LORA_HIGH']='...' / LOW 即可,优先级最高。
def _find_char(tag):
    import re
    # 先找后端规范名(char_loras 里 *_wan_t2v_{tag},最可信);找不到再退 ai-toolkit 原始 *_{tag}_noise(含训练输出)。
    groups = [[LORA_DIR + f'/**/*_wan_t2v_{tag}.safetensors'],
              [LORA_DIR + f'/**/*_{tag}_noise.safetensors',
               f'/content/ComfyUI/models/loras/**/*_{tag}_noise.safetensors',
               f'/content/mirage/*/.agent/lora_train/**/*_{tag}_noise.safetensors']]
    for pats in groups:
        fs = []
        for p in pats:
            fs += glob.glob(p, recursive=True)
        fs = [f for f in fs if 'lightx2v_t2v_lora' not in f                   # 排蒸馏目录
              and os.path.basename(f) not in ('high_noise_model.safetensors', 'low_noise_model.safetensors')
              and not re.search(r'_\d{4,}', os.path.basename(f))]            # 排带步数检查点(_000001000_);不误伤 zq+hex 触发词
        if fs:
            return sorted(set(fs), key=os.path.getmtime)[-1]                  # 该组里取最新
    return ''

def _persist(src, tag):
    if not src or src.startswith(LORA_DIR):
        return src
    tid = src.split('lora_train/')[-1].split('/')[0] if 'lora_train/' in src else 'char'
    dst = os.path.join(LORA_DIR, f'{tid}_wan_t2v_{tag}.safetensors')
    try:
        shutil.copy(src, dst); print('  已拷到 Drive 持久:', dst); return dst
    except Exception as ex:
        print('  拷贝失败,直接用原路径:', ex); return src

char_hi = _persist(os.environ.get('WAN_T2V_LORA_HIGH', '').strip() or _find_char('high'), 'high')  # 手动/自动找到的都拷 Drive 持久
char_lo = _persist(os.environ.get('WAN_T2V_LORA_LOW', '').strip()  or _find_char('low'), 'low')
print('角色 LoRA high:', char_hi or '❌(没找到→先在前端训好,或手动设 WAN_T2V_LORA_HIGH)')
print('角色 LoRA low :', char_lo or '❌(没找到)')

# 2) 角色强度(默认1.0;脸偏淡再升1.2,别一上来1.8会过曝/糊)。蒸馏固定1.0。
chi = float(os.environ.get('WAN_T2V_LORA_STR_HIGH', '1.0') or 1.0)
clo = float(os.environ.get('WAN_T2V_LORA_STR_LOW',  '1.0') or 1.0)
CFG = os.environ.get('LX_CONFIG') or '/content/wan_moe_t2v_use.json'
c = json.load(open(CFG))
_hq = os.environ.get('LIGHTX2V_HQ', '0') == '1'   # ★高画质档:不挂蒸馏+多步+开CFG(治复杂动作脱模/糙,慢但细);默认0=极速蒸馏档
triples = [('high_noise_model', char_hi, chi), ('low_noise_model', char_lo, clo)]
if not _hq:        # 蒸馏=拿画质换速度=脱模/糊的根源;高画质档去掉它
    triples += [('high_noise_model', os.environ.get('LIGHTX2V_DISTILL_LORA_HIGH', '').strip(), 1.0),
                ('low_noise_model',  os.environ.get('LIGHTX2V_DISTILL_LORA_LOW', '').strip(), 1.0)]
loras = [{'name': n, 'path': p, 'strength': s} for n, p, s in triples if p and os.path.exists(p)]
has_char = bool(char_hi and os.path.exists(char_hi) and char_lo and os.path.exists(char_lo))
if not loras and not _hq:
    print('⚠ 没找到任何 LoRA。蒸馏应由 §4 下好;角色 LoRA 先在前端训好。')
else:
    c['lora_configs'] = loras
    c['dit_quantized'] = False
    c['infer_steps'] = int(os.environ.get('LIGHTX2V_STEPS') or (20 if _hq else 8))   # 极速/成片默认8;高画质档(LIGHTX2V_HQ=1)默认20
    _NF = int(os.environ.get('LIGHTX2V_NUM_FRAMES', '81') or 81)          # 时长:81≈5s/121≈7.5s/161≈10s(须4n+1)
    for _k in ('target_video_length', 'num_frames', 'video_length', 'num_frame', 'target_frames'):
        if _k in c: c[_k] = _NF
    c['enable_cfg'] = bool(_hq)            # 高画质档开 CFG(更贴提示、结构更稳、压脱模);蒸馏档必须关
    c['sample_guide_scale'] = [5.0, 5.0] if _hq else [1.0, 1.0]   # MoE 双专家各取[0]/[1],必须2元素列表
    print('画质档 =', ('高画质/无蒸馏 ' + str(c['infer_steps']) + ' 步+CFG(慢但细,治脱模)') if _hq else ('极速/成片 ' + str(c['infer_steps']) + ' 步/蒸馏(快);脱模/想更细→设 LIGHTX2V_HQ=1 重跑'))
    CFG_LORA = '/content/wan_moe_t2v_lora.json'; json.dump(c, open(CFG_LORA, 'w'), indent=2)
    os.environ['LX_CONFIG'] = CFG_LORA
    print('写入 lora_configs:\n' + json.dumps(loras, ensure_ascii=False, indent=2))
    if not has_char:
        print('⚠ 没挂角色 LoRA → 出片不像你训的人。' + ('(高画质档跑纯底模:验画质用,正常)' if _hq else '(只蒸馏)先确认角色 LoRA 路径再跑。'))
    # 3) ★先腾显存:杀训练(run.py/ai-toolkit)+旧 server,否则起 server 加载 T5 时 OOM(训练和出片不能同时占卡)
    subprocess.run("pkill -9 -f run.py 2>/dev/null; pkill -9 -f ai-toolkit 2>/dev/null; "
                   "fuser -k 8189/tcp 2>/dev/null; pkill -9 -f 'lightx2v.server' 2>/dev/null; sleep 3; true", shell=True)
    print('已腾显存。显存:',
          subprocess.run('nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip())
    MODEL = os.environ.get('LIGHTX2V_MODEL_T2V', '/content/wan_local')
    e = dict(os.environ); e['CUDA_VISIBLE_DEVICES'] = '0'; e['PYTHONUNBUFFERED'] = '1'
    open('/content/lightx2v.log', 'w').close()
    subprocess.Popen(['python', '-u', '-m', 'lightx2v.server', '--model_cls', 'wan2.2_moe', '--task', 't2v',
                      '--model_path', MODEL, '--config_json', CFG_LORA, '--host', '0.0.0.0', '--port', '8189'],
                     cwd=LX, env=e, stdout=open('/content/lightx2v.log', 'w'), stderr=subprocess.STDOUT, start_new_session=True)
    ok = persist.wait_http('http://127.0.0.1:8189/v1/service/status', timeout=300)
    print('✅ 带 LoRA 重起就绪' if ok else '✗ 未就绪 → 跑 §5c 看日志(OOM?显存没腾干净?key不匹配?)')
    # 4) ★自检:角色 LoRA 有没有挂 + key 有没有对上
    chk = subprocess.run("grep -iE 'lora.*(not loaded|missing|mismatch|skip|unexpected|0 keys)' /content/lightx2v.log",
                         shell=True, capture_output=True, text=True).stdout.strip()
    print('—— LoRA 加载自检 ——')
    if not has_char:
        print('⚠ 没挂角色 LoRA(只蒸馏)——出片不会像你的人,先解决角色 LoRA 路径。')
    print('⚠ 疑似 key 没对上:\n' + chk if chk else '✅ 无 lora 不匹配告警(应已加载)。出片若仍像路人,把本格输出发我。')


In [ ]:
# === §6 装后端依赖 + 写 .env（纯 t2v：COMFYUI 空、lightx2v 接通）===
import os, re, shutil
%cd /content/mirage
!pip -q install -r requirements.txt
!pip -q install fastembed edge-tts
shutil.copy('.env.colab', '.env'); env = open('.env').read()
def _se(e, k, v): return re.sub(k + r'=.*', k + '=' + v, e) if (k + '=') in e else e + ('' if e.endswith(chr(10)) else chr(10)) + k + '=' + v + chr(10)
MODEL = os.environ.get('LIGHTX2V_MODEL_T2V') or '/content/wan_local'   # §4 解析好的权重路径(本地或 Drive)
# 蒸馏 4 步 LoRA(§4 已 glob 出实际文件名) → 登记到 LIGHTX2V_DISTILL_LORA_*(不是角色槽!);角色 LoRA 走 WAN_T2V_LORA_*(训完再填)
_dhi = os.environ.get('LIGHTX2V_DISTILL_LORA_HIGH', '')
_dlo = os.environ.get('LIGHTX2V_DISTILL_LORA_LOW', '')
for k, v in [('COMFYUI_BASE_URL', ''), ('LIGHTX2V_ENABLED', 'true'), ('LIGHTX2V_BASE_URL', 'http://127.0.0.1:8189'),
             ('LIGHTX2V_MODEL_T2V', MODEL), ('T2V_PROVIDER', 'lightx2v-t2v'), ('VIDEO_PROVIDER_DEFAULT', 'wan2.2'),
             ('LIGHTX2V_DISTILL_LORA_HIGH', _dhi), ('LIGHTX2V_DISTILL_LORA_LOW', _dlo),
             ('COMFYUI_LORA_DIR', CACHE + '/char_loras'),                              # ★训好的角色 LoRA 自动存 Drive(持久,免 Colab 回收丢)
             ('LORA_TRAIN_BASE', 'ai-toolkit/Wan2.2-T2V-A14B-Diffusers-bf16'),         # Wan2.2-T2V 训练底模(不是 FLUX;前端训练用)
             ('AI_TOOLKIT_DIR', '/content/ai-toolkit')]:                                # ai-toolkit 位置(前端「开始训练」需先装它,见可选安装格)
    env = _se(env, k, v)
if DEEPSEEK_KEY:
    env = _se(env, 'OPENAI_API_KEY', DEEPSEEK_KEY)
open('.env', 'w').write(env)
os.environ['COMFYUI_LORA_DIR'] = CACHE + '/char_loras'   # ★同步进内核:§5d 找角色 LoRA 读的是 os.environ,别只靠硬编码兜底
print('.env 写好(纯 t2v: COMFYUI 空 / lightx2v 接通 / 蒸馏 LoRA 登记 LIGHTX2V_DISTILL_LORA_*)')
print('LoRA 挂载:权威路径是 §5d 把 lora_configs 写进 server config(per-request 传会被 server 忽略)。')
print('  后端 provider 接入已修(条目带 name=high/low_noise_model、前向兼容);角色 LoRA 训完填 WAN_T2V_LORA_HIGH/LOW 再跑 §5d。')

In [ ]:
# === §7 构建前端（纯 t2v 工作台）===
%cd /content/mirage/frontend
!npm install --silent && npm run build || echo '前端构建失败(下面校验)'
%cd /content/mirage
import os as _os
print('前端产物 index.html:', '✅ 已构建' if _os.path.exists('/content/mirage/mirage/static/index.html') else '❌ 没构建出来→公网URL打开会是空白(API仍可用),请重跑本格 §7')

In [ ]:
# === §8 起后端（硬重启：杀旧+起新，读 §6 的 .env）===
import sys; sys.path.insert(0, '/content/mirage/colab'); import persist
persist.restart_service('后端', ['uvicorn', 'mirage.main_api:app', '--host', '0.0.0.0', '--port', '8000'],
    'http://127.0.0.1:8000/api/health', '/content/api.log', 8000, 'uvicorn', cwd='/content/mirage', timeout=120)

In [ ]:
# === §9 cloudflared 暴露后端 → 公网 URL ===
import subprocess, re, time, os, sys
sys.path.insert(0, '/content/mirage/colab'); import persist
subprocess.run('pkill -9 -f cloudflared 2>/dev/null; sleep 2', shell=True)
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True)
open('/content/cf.log', 'w').close()
persist.start_bg(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], '/content/cf.log')
url = None
for _ in range(60):
    time.sleep(1)
    m = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cf.log').read())
    if m:
        url = m[-1]; break
print('✅ 公网地址:', url if url else '⚠ 60s 未就绪,重跑本格')
print('打开它 → 工作台纯 t2v: 粘小说 → 一键分析 → 每镜「出片(t2v)」')

In [ ]:
# === §10 实时日志（lightx2v 出片采样进度；停本格只停 tail、不杀 server）===
!tail -n 60 /content/lightx2v.log   # 非阻塞快照:Run all 不会卡在这格。实时盯片单独跑: !tail -n 80 -f /content/lightx2v.log

In [ ]:
# === §日志速查（一键体检：跑一下看各服务+日志现状；下面注释是按需单独跑的命令）===
import subprocess
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout
print('【进程】 lightx2v:', '活✅' if sh("pgrep -f lightx2v.server").strip() else '没了❌',
      '| 后端:', '活✅' if sh("pgrep -f mirage.main_api").strip() else '没了❌',
      '| cloudflared:', '活✅' if sh("pgrep -f cloudflared").strip() else '没了❌')
print('【lightx2v 状态】', (sh("curl -s -m5 http://127.0.0.1:8189/v1/service/status") or '(无响应/加载中)').strip()[:200])
print('【公网 URL】', (sh(r"grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /content/cf.log | tail -1") or '(还没有)').strip())
print('【出片真帧 REAL_TB（NoneType 调试）】')
print(sh("grep -A40 REAL_TB /content/lightx2v.log | tail -45") or '  (没有 REAL_TB → 还没崩过/还没出片)')
print('【lightx2v.log 尾 25 行】'); print(sh("tail -25 /content/lightx2v.log"))
print('【后端 api.log 尾 12 行】'); print(sh("tail -12 /content/api.log"))
# ── 按需单独跑(复制到新格)──────────────────────────
# 实时盯出片采样(会一直刷,停本格即停): !tail -n 80 -f /content/lightx2v.log
# 只挖出片真帧(NoneType 定位)        : !grep -A60 REAL_TB /content/lightx2v.log
# 挂 LoRA 后查 key 不匹配            : !grep -iE "lora.*(not loaded|missing|mismatch|skip)" /content/lightx2v.log
# 看当前生效配置(attn/boundary/offload): import json; print(json.load(open('/content/wan_moe_t2v_use.json')))
# GPU 占用 / 后端实时日志            : !nvidia-smi   /   !tail -n 60 -f /content/api.log
# 验证帧数是否 per-request 生效   : 出两条不同「时长」的片,看 api.log 的「num_frames=」与实际出片秒数是否一致;若改时长出片仍 5s=server 按 config 锁帧长,改 §3/§5d 的 LIGHTX2V_NUM_FRAMES 重起


In [ ]:
# === §SVI 验证（手动单独跑！别放进 Run all）——验证 lightx2v 能不能挂 SVI 做 i2v 续接 ===
# 目的:在投入做"续接成长片"前,先验 lightx2v --task i2v + SVI v2_PRO LoRA 能不能跑通、不崩、接得上。
# ⚠️ 探索性模板:i2v 的 --model_cls / SVI LoRA 命名 / i2v 请求体可能要按你的 lightx2v 实际版本微调。
# ⚠️ 下 Wan2.2-I2V-A14B 底模很大(~67-118G,放 Drive 持久);会先杀 t2v server 腾显存(一张卡放不下两套底模)。
import os, subprocess, sys, glob, json, time
sys.path.insert(0, '/content/mirage/colab'); import persist
from huggingface_hub import snapshot_download
LX = '/content/LightX2V'; DRIVE = '/content/drive/MyDrive'
I2V = DRIVE + '/mirage_models/wan2.2_i2v_a14b'      # i2v 底模(Drive 持久;挂 LoRA 必须 bf16,别下量化版)
SVI = DRIVE + '/mirage_models/svi_v2_pro'           # SVI v2_PRO LoRA
os.makedirs(I2V, exist_ok=True); os.makedirs(SVI, exist_ok=True)
# 1) 下 i2v 底模(bf16)
if len(glob.glob(I2V + '/high_noise_model/*.safetensors')) < 6:
    print('下 Wan2.2-I2V-A14B 到 Drive(大,~10-20 分钟,断点续传)...')
    snapshot_download('Wan-AI/Wan2.2-I2V-A14B', local_dir=I2V, max_workers=8,
        allow_patterns=['high_noise_model/*', 'low_noise_model/*', '*.json', 'Wan2.1_VAE.pth',
                        'models_t5_umt5-xxl-enc-bf16.pth', 'models_clip*', 'google/*'])
# 2) 下 SVI v2_PRO 两个 LoRA(要带 v2_PRO + A14B,别拿成非 Pro/2.1 版)
snapshot_download('Kijai/WanVideo_comfy', local_dir=SVI,
                  allow_patterns=['*Stable-Video-Infinity*v2*PRO*A14B*'])
svi_hi = next(iter(sorted(glob.glob(SVI + '/**/*PRO*HIGH*.safetensors', recursive=True))), '')
svi_lo = next(iter(sorted(glob.glob(SVI + '/**/*PRO*LOW*.safetensors', recursive=True))), '')
print('SVI high/low:', svi_hi or '❌ 没找到(检查仓库结构/文件名)', '/', svi_lo or '❌')
# 3) 先杀 t2v server 腾显存,再起 lightx2v --task i2v 挂 SVI(用 §3 的 use.json 当基配:含 rope/attn 修复)
subprocess.run("pkill -9 -f run.py 2>/dev/null; fuser -k 8189/tcp 2>/dev/null; pkill -9 -f 'lightx2v.server' 2>/dev/null; sleep 3; true", shell=True)
c = json.load(open(os.environ.get('LX_CONFIG') or '/content/wan_moe_t2v_use.json'))
c['lora_configs'] = [{'name': 'high_noise_model', 'path': svi_hi, 'strength': 1.0},
                     {'name': 'low_noise_model',  'path': svi_lo, 'strength': 1.0}]
c['dit_quantized'] = False
c['infer_steps'] = 6                       # SVI 社区:6 步 / CFG1.5 / shift8(蒸馏路线)
c['enable_cfg'] = True
c['sample_guide_scale'] = [1.5, 1.5]
CFG_I2V = '/content/wan_moe_i2v_svi.json'; json.dump(c, open(CFG_I2V, 'w'), indent=2)
e = dict(os.environ); e['CUDA_VISIBLE_DEVICES'] = '0'; e['PYTHONUNBUFFERED'] = '1'; open('/content/lightx2v.log', 'w').close()
subprocess.Popen(['python', '-u', '-m', 'lightx2v.server', '--model_cls', 'wan2.2_moe', '--task', 'i2v',
                  '--model_path', I2V, '--config_json', CFG_I2V, '--host', '0.0.0.0', '--port', '8189'],
                 cwd=LX, env=e, stdout=open('/content/lightx2v.log', 'w'), stderr=subprocess.STDOUT, start_new_session=True)
ok = persist.wait_http('http://127.0.0.1:8189/v1/service/status', timeout=600)
print('i2v+SVI server:', '✅ 就绪' if ok else '✗ 没起来 → 跑 §5c 看日志(--task i2v 不认? --model_cls 要换? SVI key 不匹配?)')
print('—— 验通了才谈接进产品 ——')
print('① 自检 SVI key 有没有挂上:  !grep -iE "lora.*(not loaded|missing|mismatch|0 keys)" /content/lightx2v.log')
print('② 出一条续接片:准备一张首帧图(胖哥正脸),POST /v1/tasks/ 带 image_path=该图、prompt=运镜,看接不接得上、崩不崩、糊不糊。')
print('③ 全跑通(挂上+出片+接得上) → 告诉我,我再帮你落「t2v 出首帧 → i2v+SVI 续段」整条管线 + 前端入口。')
print('④ 验完恢复 t2v:重跑 §5(或 §5d 挂角色 LoRA)即可(i2v server 已被本格起的占着,§5 会硬重启换回 t2v)。')


In [ ]:
# === §Stand-In 强锁脸（手动单独跑；给一张参考脸 → 跨镜锁同一张脸，免训练）===
# 它另起一套引擎(DiffSynth,不走 lightx2v),复用你 §4 下的 Wan2.2 权重当 base(不重下底模),端口 8190。
# 用法:起好后 → 前端「角色 & LoRA」给主角传一张清晰正脸 → 出片勾「强锁脸(Stand-In)」→ 该角色镜走锁脸通道。
# ⚠️ 与 lightx2v 同卡共存:两套 A14B 在 96G 上偏紧。若 OOM 且你只做锁脸:先停 lightx2v → !pkill -9 -f lightx2v.server
import os, sys, glob, subprocess, re
sys.path.insert(0, '/content/mirage/colab'); import persist
SI = '/content/Stand-In'; VENV = '/content/standin_venv'; PY = VENV + '/bin/python'
BASE = '/content/wan_local'   # §4 下的 Wan2.2-T2V-A14B;布局(high/low_noise_model+T5+VAE+google/)正好匹配 Stand-In base_path
if not (len(glob.glob(BASE + '/high_noise_model/*.safetensors')) >= 6 and len(glob.glob(BASE + '/low_noise_model/*.safetensors')) >= 6):
    print('❌ 没找到 /content/wan_local 的 Wan2.2 权重 → 先把 §4 跑完(下底模)再回来跑本格。')
else:
    # 1) clone Stand-In
    if not os.path.isdir(SI + '/.git'):
        subprocess.run('git clone https://github.com/WeChatCV/Stand-In ' + SI, shell=True)
    # 2) venv 复用系统 torch(--system-site-packages,免重下~6G);装 Stand-In 依赖(去掉 torch 行,避免覆盖 cu128 torch)+ fastapi/uvicorn
    if not os.path.exists(PY):
        subprocess.run('python -m venv --system-site-packages ' + VENV, shell=True)
    reqs = SI + '/requirements.txt'
    if os.path.exists(reqs):
        keep = [ln for ln in open(reqs) if ln.strip() and not re.match(r'\s*torch', ln, re.I)]
        open('/content/standin_reqs.txt', 'w').writelines(keep)
        subprocess.run(PY + ' -m pip install -q -r /content/standin_reqs.txt', shell=True)
    subprocess.run(PY + ' -m pip install -q fastapi uvicorn', shell=True)
    # 3) 下适配器(BowenXue/Stand-In)+ antelopev2 到 Stand-In 的 checkpoints 约定路径
    from huggingface_hub import snapshot_download
    snapshot_download('BowenXue/Stand-In', local_dir=SI + '/checkpoints/Stand-In')
    snapshot_download('DIAMONIK7777/antelopev2', local_dir=SI + '/checkpoints/antelopev2/models/antelopev2')
    # 4) 起 server(venv python),env 指好 base/antelope;脱离内核(停 cell 不杀)
    subprocess.run("fuser -k 8190/tcp 2>/dev/null; pkill -9 -f standin_server.py 2>/dev/null; sleep 2; true", shell=True)
    e = dict(os.environ)
    e.update({'STANDIN_DIR': SI, 'STANDIN_BASE_PATH': BASE, 'STANDIN_WAN_VERSION': '2.2',
              'STANDIN_ANTELOPE': SI + '/checkpoints/antelopev2', 'STANDIN_PORT': '8190',
              'CUDA_VISIBLE_DEVICES': '0', 'PYTHONUNBUFFERED': '1'})
    open('/content/standin.log', 'w').close()
    subprocess.Popen([PY, '/content/mirage/colab/standin_server.py'], env=e,
                     stdout=open('/content/standin.log', 'w'), stderr=subprocess.STDOUT, start_new_session=True)
    ok = persist.wait_http('http://127.0.0.1:8190/v1/health', timeout=900)
    print('Stand-In server:', '✅ 就绪' if ok else '✗ 没起来 → !tail -60 /content/standin.log 看错(多半依赖/适配器没下全或显存不足)')
    # 5) 写 .env(开 STANDIN_*)+ 重启后端,让 standin-t2v provider 注册
    %cd /content/mirage
    env = open('.env').read()
    def _se(s, k, v):
        return re.sub(k + r'=.*', k + '=' + v, s) if (k + '=') in s else s + ('' if s.endswith(chr(10)) else chr(10)) + k + '=' + v + chr(10)
    for k, v in [('STANDIN_ENABLED', 'true'), ('STANDIN_BASE_URL', 'http://127.0.0.1:8190'), ('STANDIN_STEPS', '20')]:
        env = _se(env, k, v)
    open('.env', 'w').write(env)
    persist.restart_service('后端', ['uvicorn', 'mirage.main_api:app', '--host', '0.0.0.0', '--port', '8000'],
        'http://127.0.0.1:8000/api/health', '/content/api.log', 8000, 'uvicorn', cwd='/content/mirage', timeout=120)
    print('✅ Stand-In 接通。前端:给主角传参考脸 → 出片勾「强锁脸(Stand-In)」。该角色镜走锁脸(无蒸馏~20步,较慢);其余镜仍走 lightx2v(快)。')
    print('   排错:!tail -60 /content/standin.log  | 显存紧:!pkill -9 -f lightx2v.server(只做锁脸时)')


In [ ]:
# === §i2v续接(手动跑;尾帧续接:镜1链头 → 镜2.. 接上一镜尾帧 i2v 续生成)===
# 轮流:停 t2v 腾显存、起 i2v server。i2v 底模在 Drive(回收后仍在);HF 缓存钉死 Drive,不漏本地(防爆盘)。
# ★实验旋钮(2026-06-20 调研结论):挂的胖哥 LoRA 是【t2v 训的】,用在 i2v 上是错配——强度>1 反而毁脸/降一致。
#   所以默认 0.5(错配 LoRA 降强度试);想对照设 0.0(完全不挂、纯靠首帧);将来训出 i2v 原生 LoRA 再回 1.0。
I2V_LORA_STRENGTH = 0.5     # ← 0.5 试 / 0.0 不挂对照 / 1.0(仅当用 i2v 原生 LoRA)
import os, sys, glob, json, subprocess, re, shutil, importlib
sys.path.insert(0, '/content/mirage/colab'); import persist
LX = '/content/LightX2V'; CACHE = '/content/drive/MyDrive/mirage_models'
HFCACHE = '/content/drive/MyDrive/mirage_tools/hf_cache'; os.makedirs(HFCACHE, exist_ok=True)
os.environ['HF_HOME'] = HFCACHE; os.environ['HF_HUB_CACHE'] = HFCACHE + '/hub'   # ★缓存锁 Drive,不漏本地(爆盘元凶)
LOCAL_I2V = '/content/wan_i2v_local'; DRIVE_I2V = CACHE + '/wan2.2_i2v_a14b'; os.makedirs(DRIVE_I2V, exist_ok=True)
def _complete(d): return len(glob.glob(d + '/high_noise_model/*.safetensors')) >= 6 and len(glob.glob(d + '/low_noise_model/*.safetensors')) >= 6
I2V = LOCAL_I2V if _complete(LOCAL_I2V) else DRIVE_I2V     # 跑过 §i2v搬本地 就用本地秒载,否则 Drive
_df = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
if not _complete(I2V) and _df < 130:
    raise SystemExit(f'❌ Drive 只剩 {_df:.0f}G < 130G(i2v 需 126G)。先清:!rm -rf {HFCACHE}  再重跑本格')
if not _complete(I2V):
    from huggingface_hub import snapshot_download
    print(f'Drive 无完整 i2v(空 {_df:.0f}G)→ 下 Wan2.2-I2V-A14B 到 Drive(~126G,断点续传,只此一次)...')
    snapshot_download('Wan-AI/Wan2.2-I2V-A14B', local_dir=DRIVE_I2V, cache_dir=HFCACHE, max_workers=8,
        allow_patterns=['high_noise_model/*', 'low_noise_model/*', '*.json', 'Wan2.1_VAE.pth', 'models_t5_umt5-xxl-enc-bf16.pth', 'google/*'])
    I2V = DRIVE_I2V
print('i2v 底模:', I2V, '(本地·秒载)' if I2V == LOCAL_I2V else '(Drive·慢~25min,想快先跑 §i2v搬本地)')
# i2v 基配 + 同 §3 修复(rope/attn) + 角色 LoRA(可调强度) + 帧长
_cands = sorted(glob.glob(f'{LX}/configs/**/wan_moe_i2v*.json', recursive=True))
src_cfg = next((x for x in _cands if 'distill' not in x.lower() and 'nvfp4' not in x.lower()), (_cands[0] if _cands else f'{LX}/configs/wan22/wan_moe_i2v.json'))
c = json.load(open(src_cfg))
_attn = 'sage_attn2' if subprocess.run('python -c "import sageattention" 2>/dev/null', shell=True, capture_output=True).returncode == 0 else 'torch_sdpa'
for k in ('self_attn_1_type', 'cross_attn_1_type', 'cross_attn_2_type'): c[k] = _attn
c['rope_type'] = 'torch'; c['rms_norm_type'] = 'torch'; c['dit_quantized'] = False
c['cpu_offload'] = (os.environ.get('LX_CPU_OFFLOAD', 'false') == 'true')
LORA_DIR = os.environ.get('COMFYUI_LORA_DIR') or (CACHE + '/char_loras')
def _find(tag):
    fs = [f for f in glob.glob(LORA_DIR + f'/**/*_wan_t2v_{tag}.safetensors', recursive=True) + glob.glob(LORA_DIR + f'/**/*_{tag}_noise.safetensors', recursive=True) if 'lightx2v_t2v_lora' not in f and not re.search(r'_\d{4,}', os.path.basename(f))]
    return sorted(set(fs), key=os.path.getmtime)[-1] if fs else ''
chi, clo = _find('high'), _find('low')
if I2V_LORA_STRENGTH > 0 and chi and clo:
    c['lora_configs'] = [{'name': 'high_noise_model', 'path': chi, 'strength': I2V_LORA_STRENGTH},
                         {'name': 'low_noise_model', 'path': clo, 'strength': I2V_LORA_STRENGTH}]
    print(f'角色 LoRA:挂 {os.path.basename(chi)} 强度 {I2V_LORA_STRENGTH}(t2v 训的→错配,故低强度)')
else:
    c.pop('lora_configs', None)
    print('角色 LoRA:不挂(I2V_LORA_STRENGTH=0 对照,或没找到 LoRA)——纯靠首帧锁身份')
_NF = int(os.environ.get('LIGHTX2V_NUM_FRAMES', '81') or 81)
for _k in ('target_video_length', 'num_frames', 'video_length', 'num_frame', 'target_frames'):
    if _k in c: c[_k] = _NF
CFG_I2V = '/content/wan_moe_i2v_use.json'; json.dump(c, open(CFG_I2V, 'w'), indent=2)
print('i2v 配置:', src_cfg.replace(LX + '/', ''), '| attn:', _attn, '| 步数:', c.get('infer_steps'))
# ★轮流:停 t2v 腾显存,起 i2v on 8190
subprocess.run("pkill -9 -f 'lightx2v.server' 2>/dev/null; pkill -9 -f standin_server.py 2>/dev/null; fuser -k 8189/tcp 2>/dev/null; fuser -k 8190/tcp 2>/dev/null; sleep 4; true", shell=True)
print('已停 t2v。显存:', subprocess.run('nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip())
e = dict(os.environ); e['CUDA_VISIBLE_DEVICES'] = '0'; e['PYTHONUNBUFFERED'] = '1'
subprocess.Popen(['python', '-u', '-m', 'lightx2v.server', '--model_cls', 'wan2.2_moe', '--task', 'i2v', '--model_path', I2V, '--config_json', CFG_I2V, '--host', '0.0.0.0', '--port', '8190'],
                 cwd=LX, env=e, stdout=open('/content/lightx2v_i2v.log', 'w'), stderr=subprocess.STDOUT, start_new_session=True)
print('i2v server 启动中…(本地~3min / Drive~25min)。另开格 !tail -f /content/lightx2v_i2v.log 看进度')
ok = persist.wait_http('http://127.0.0.1:8190/v1/service/status', timeout=2400)   # 等够 40 分钟(Drive 慢,别提前放弃)
print('i2v server:', '✅ 就绪' if ok else '✗ !tail -40 /content/lightx2v_i2v.log')
# 写 .env(开 i2v + 指 model_path)+ 重启后端注册 lightx2v-i2v provider + 续接路由
%cd /content/mirage
env = open('.env').read()
def _se(s, k, v): return re.sub(k + r'=.*', k + '=' + v, s) if (k + '=') in s else s + ('' if s.endswith(chr(10)) else chr(10)) + k + '=' + v + chr(10)
for k, v in [('LIGHTX2V_I2V_ENABLED', 'true'), ('LIGHTX2V_I2V_BASE_URL', 'http://127.0.0.1:8190'), ('LIGHTX2V_MODEL_I2V', I2V)]:
    env = _se(env, k, v)
open('.env', 'w').write(env)
importlib.reload(persist)
persist.restart_service('后端', ['uvicorn', 'mirage.main_api:app', '--host', '0.0.0.0', '--port', '8000'], 'http://127.0.0.1:8000/api/health', '/content/api.log', 8000, 'uvicorn', cwd='/content/mirage', timeout=120)
print('✅ i2v 续接接通。前端「🔗 续接出片」/每镜「🔗续这镜」可用。改 I2V_LORA_STRENGTH 重跑本格即换强度。')
print('   前置:镜1 要先有 t2v 成片当链头(t2v 已删则需 §4/§5 重起 t2v 先出镜1,或上传一段当链头)。')


In [ ]:
# === §i2v搬本地(可选·加速;把 i2v DiT 从 Drive 拷到本地,以后每次 §i2v续接 秒载)===
# 前提:§i2v续接 已把 i2v 下到 Drive。一次性拷 ~25min,之后重启从 ~25min → ~3min。中断重跑本格(rsync 续传)。
# 本地装不下会直接拦下、不爆盘。搬完重跑 §i2v续接,它会自动用本地。
import shutil, subprocess, os, glob, time
DRIVE_I2V = '/content/drive/MyDrive/mirage_models/wan2.2_i2v_a14b'; LOCAL_I2V = '/content/wan_i2v_local'
subprocess.run("pkill -9 -f 'lightx2v.server' 2>/dev/null; fuser -k 8190/tcp 2>/dev/null; sleep 4; true", shell=True)
dit_kb = int(subprocess.run(f"du -sk {DRIVE_I2V}/high_noise_model {DRIVE_I2V}/low_noise_model 2>/dev/null | awk '{{s+=$1}}END{{print s}}'", shell=True, capture_output=True, text=True).stdout.strip() or 0)
dit_g = dit_kb / 1e6; free_g = shutil.disk_usage('/content').free / 1e9
print(f'DiT {dit_g:.0f}G | 本地空 {free_g:.0f}G')
if dit_g < 1:
    raise SystemExit('❌ Drive 上没找到 i2v DiT。先跑 §i2v续接 把 i2v 下到 Drive,再搬。')
if dit_g + 8 > free_g:
    raise SystemExit(f'❌ 本地空间不稳(空 {free_g:.0f}G,DiT 要 {dit_g:.0f}G+8G余量)。别搬,继续用 Drive。')
os.makedirs(LOCAL_I2V, exist_ok=True)
print('rsync 拷 DiT 到本地…(~25min,可断点续传)'); t0 = time.time()
subprocess.run(f"rsync -a {DRIVE_I2V}/high_noise_model {DRIVE_I2V}/low_noise_model {LOCAL_I2V}/", shell=True)
for f in os.listdir(DRIVE_I2V):                       # T5/VAE/json/google 软链(小,留 Drive)
    if f in ('high_noise_model', 'low_noise_model'): continue
    dst = os.path.join(LOCAL_I2V, f)
    if not os.path.exists(dst):
        try: os.symlink(os.path.join(DRIVE_I2V, f), dst)
        except FileExistsError: pass
nh = len(glob.glob(LOCAL_I2V + '/high_noise_model/*.safetensors')); nl = len(glob.glob(LOCAL_I2V + '/low_noise_model/*.safetensors'))
print(f'拷完 {int((time.time()-t0)/60)}min | 本地 DiT high={nh} low={nl} | 本地剩 {shutil.disk_usage("/content").free/1e9:.0f}G')
print('✅ 搬完。重跑 §i2v续接 即自动用本地、秒载。' if nh >= 6 and nl >= 6 else '⚠️ 不完整,重跑本格续传')


In [ ]:
# === §项目备份/恢复(防回收丢项目;手动跑)===
# Colab 回收清空 /content → 项目(分镜+成片,在 /content/mirage/mirage_workspace)会丢。
# 用法:① 关 Colab 前 MODE='backup' 跑一次 ② 新开 Colab、起后端(§8)【之前】MODE='restore' 跑一次。
# 只 rsync 文件(sqlite DB 全程在本地、不走 Drive FUSE,避免锁/损坏),安全。
MODE = 'backup'        # ← 'backup' 备份到 Drive / 'restore' 从 Drive 恢复(起后端前)
import subprocess, os
WS = '/content/mirage/mirage_workspace'; BAK = '/content/drive/MyDrive/mirage_workspace_backup'
os.makedirs(BAK, exist_ok=True); os.makedirs(WS, exist_ok=True)
if MODE == 'backup':
    subprocess.run(f"rsync -a --delete {WS}/ {BAK}/", shell=True)
    print('✅ 已备份项目到 Drive:', BAK, '(下次新开 Colab 用 restore 拿回)')
elif MODE == 'restore':
    subprocess.run(f"rsync -a {BAK}/ {WS}/", shell=True)
    print('✅ 已从 Drive 恢复项目到', WS, '——现在再跑 §8 起后端,刷新前端就能看到旧项目')
else:
    print('MODE 只能是 backup 或 restore')


---
## 🎉 经验总结（RTX PRO 6000 实测跑通）

**一句话**：lightx2v 装机坑 §3 全固化了；出片成败的关键是**别让任何算子默认到没装的后端**——`attn` 三键 + **`rope_type`** 都得强制 `torch`，否则报 `'NoneType' object is not callable`。

**验证硬件**：RTX PRO 6000（Blackwell sm120, 96G）→ `cpu_offload=False`、双专家全 GPU bf16；SageAttention 在 sm120 编不过（已知：用 Hopper wgmma 指令）→ 自动退 `torch_sdpa`，不影响出片。

### 安装坑（§3 已自动处理，不用手动改）
1. lightx2v 0.1.0 钉 `torch<2.8.0`，撞 Colab cu128 torch2.8 → 本体&依赖全 `--no-deps` 装；**别 `pip install lightx2v`**（PyPI 是残壳，装前先 uninstall + editable）。
2. editable `.pth` 当前内核不重读 → 一切以**干净子进程** import 为准。
3. `worldmirror`（硬 import flash_attn）/ `ring_attn`（`@torch.jit.script` 新 torch 编不过）跟单卡 t2v 无关却崩 import → patch 文件。
4. torch↔torchvision ABI 不匹配 → 子进程探测、坏了原子重装匹配的 cu128 三件套。

### 配置坑（出片成败的关键，§3 已固化）
5. **必须用 `configs/wan22/wan_moe_t2v.json`**（含 `boundary=0.875` 双专家切换点）；`wan/wan_t2v.json` 是旧单模型 → 崩 `KeyError 'boundary'`。
6. **`attn` 三键**（`self_attn_1_type`/`cross_attn_1_type`/`cross_attn_2_type`）→ `torch_sdpa`（`flash_attn3` 没装）。
7. **★`rope_type`** 默认 `flashinfer`（没装）→ `apply_rope...` 是 `None`，第一个 transformer block 调它就崩。**必须 `rope_type=torch`**——这是「attn 都改了仍崩」的最后一把钥匙；`rms_norm_type=torch` 防御性同钉。
8. 量化默认关（bf16）；挂 LoRA 与量化互斥，别同时开 fp8。
9. **★4 步极速档（出片提速 ~10×）**：挂蒸馏 LoRA + `infer_steps=4` + `enable_cfg=False` + **`sample_guide_scale` 必须是 `[v, v]` 两元素列表**（MoE 双专家各取一个 `[0]`/`[1]`；设成标量 `1.0` 会崩 `'float' object is not subscriptable` @ wan_runner.py get_current_model_index）。§5d 已固化；RTX PRO 6000 实测跑通。
10. **取片**：lightx2v 把片存 `<install>/lightx2v/server_cache/outputs/{task_id}.mp4`（同机本地）；mirage 已按此约定拷回（commit 02dda60），前端正常收片。

### 权重 / 加载
9. `snapshot_download` 直接下本地 `/content/wan_local`（比 Drive FUSE 快几倍，本地 SSD 加载秒级）；代价=每会话重下 ~67G。
10. diffusers 格式直读；只有崩 `patch_embedding` 才跑 §5b 转 native。

### 出片报错怎么挖真帧
11. 出片 `'NoneType' object is not callable`：`base.py:196` 是**重抛**、真帧跨进程丢；**真帧在 worker** → `grep -n 'inference failed' -A 120 /content/lightx2v.log`（`worker.py:108 logger.exception` 打的，在 `REAL_TB` **之前**）。看最深帧 `File ... line ...` 定位是哪个没装的算子，按同法把对应 `*_type` 强制 `torch`。

---
## 角色 LoRA 训练 & 挂载（Wan-T2V，全自动）
- **训练**：在前端 Web 工作台「角色 & LoRA」里上传脸照训练（务必含脸部特写）；产物自动落 Drive `char_loras/`，回收不丢。
- **挂载**：跑 **§5d** 即可——它【自动】从 Drive `char_loras/` 找到**最新**训好的角色 LoRA、挂上并重起 server，**无需手动填路径**。没训过角色 LoRA 时只挂蒸馏(提速)，会打印提示。
  - LoRA **只在「起 server 的 config」里生效**（per-request 传会被忽略）；**改 LoRA/换角色 → 重跑 §5d**；**挂 LoRA 必须 bf16，§5d 已自动关量化**。
  - 想指定某个角色 LoRA（而非最新）：跑 §5d 前 `os.environ['WAN_T2V_LORA_HIGH'/'LOW']=...`（优先级最高）。


### ★出片「跟训练的人完全对不上 / 是别人」——头号坑：触发词
角色 LoRA 身份能否绑上，命门是**触发词必须是无预训练语义的罕见 token**。用 `char`/`person`/人名 这类常见词当触发词，
身份会糊进该词原有语义 → 出片渲染的是预训练里的「某个人」而非你训的人（症状：完全是别人）。
**后端已自动修这两点（2026-06-18）**，前端无需再操心：
1. 触发词留空或填了常见词 → 自动换成 `zq` 开头的罕见 token（`effective_trigger`）；打 caption 与出片注入统一用它。
2. 打 caption **只写触发词、绝不写外貌**（脸写进 caption 会和身份绑定打架）。外貌只在出片画面提示词里出现。
> 旧 LoRA（触发词 `char` + caption 带外貌）已被污染、救不回 → **必须重训**：前端「角色 & LoRA」重新上传同一批图（选中该角色，触发词留空），重训后跑 §5d 挂上。

### 仍偏淡 / 像但会飘 —— 是 T2V-LoRA 的天花板，不是 bug
- 干净配方下 Wan2.2-T2V 角色 LoRA 给的是**强可辨识度**（一看就是这个人），但**运镜/转头/大角度时脸会朝通用脸漂**——这是 T2V 角色 LoRA 的固有局限。
- 想更稳：① 出片用**中等质量档（infer_steps 6-8）**而非 4 步极速档（给身份更多步数，比硬拉强度稳）；② 角色强度在 §5d 用 `WAN_T2V_LORA_STR_HIGH/LOW` 微调（淡→1.1~1.2，变形→0.7~0.9）；③ 数据集凑到 ~25-30 张、多角度多光照。
- 要**精确到像素的同一张脸**：T2V-LoRA 做不到，那是图像模型（FLUX+PuLID）/ i2v 参考图 / 换脸后处理的领域。

## 画质 / 显存 / 多人串脸（2026-06-19 补）
- **画质 = 采样步数(infer_steps)，跟 GPU 无关**：96G 和 5090 跑同一模型同设置，画质一样；大卡只换速度/规模/能训练，不换每帧画质。档位：极速 4 步(打样、糙) / 成片 8 步(默认) / 精修 16 步 / 最高=**关蒸馏跑 20-30 步 + 开 CFG**(慢但最细)。前端「画质档」走 per-request steps；切档没变化=该 server 版本只认起 server 的 config → 改 §5d 的 `LIGHTX2V_STEPS` 重起。
- **显存**：Wan2.2-T2V-A14B = MoE 双专家 + umt5-xxl，bf16 不量化(挂 LoRA 必须 bf16)→ 加载 ~40G、出片爬到 ~67-83G，**96G 正常、不用慌**。唯一禁忌：别一边训练(ai-toolkit)一边出片(lightx2v)——抢同一张卡会 OOM(§5d 已自动先杀训练腾显存)。
- **多人串脸**：t2v 全局角色 LoRA 会把同框里**所有人**都画成主角的脸(机理硬伤、调参只降概率不能归零)。后端已缓解(多人镜不注入触发词 + 自动加负向)；要稳定出「主角遇见另一个不同的人」，就**拆成单人镜 + 剪辑制造同框**，别让两人同框同生成。

## 首跑核对清单
1. **§1** 打印 `GPU ... smX.Y ...G -> cpu_offload=...`？没 GPU → 运行时改选 GPU。
2. **§3** 末行 `lightx2v import: ✅ OK | 配置: configs/wan22/wan_moe_t2v.json | boundary: 0.875 | attn: torch_sdpa`（且 §3 已强制 `rope_type=torch`——出片不崩 NoneType 的关键）？`SageAttention: False` 不影响。
3. **§4** `本地权重 high/low: 6 / 6 ✅` + `蒸馏 LoRA high/low: ...`？否 → 重跑 §4（断点续传）。
4. **§5** `✅ lightx2v 就绪`？先验**裸片**（无 LoRA）出片通不通。
5. **§6 → §8 → §9** .env 接通 → 后端起 → 公网 URL → 出片。
6. 裸片通了、想锁人物 → 训角色 LoRA → 填进 **§5d** 跑一次（重起 server）。